# Checkpoint 1: Credit Risk Baseline

## Objective

Build a reproducible, leakage-safe baseline model to estimate whether a loan application will **default**.

The notebook covers:
1. Data loading and schema validation
2. Data inspection and cleaning
3. Exploratory data analysis (EDA)
4. Feature preparation (leakage-safe)
5. Model training (Logistic Regression baseline)
6. Evaluation: ROC AUC, Gini, Accuracy, Precision, Recall, F1, Confusion Matrix
7. Conclusion

**Dataset**: `finwise_loan_applications.csv` — 1 000 synthetic loan applications for a FinTech lender.

**Target**: `default_flag` (0 = no default, 1 = default)

**Key design choice**: All learned transformations (scaling, one-hot encoding) are fitted on the *training split only*, so no test-set information leaks into model training.

---
## 1. Data Loading and Inspection

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data.load_data import load_loan_data
from src.data.preprocess import clean_loan_data, split_features_target
from src.models.train_model import build_model_pipeline, evaluate_model, RANDOM_STATE

# Path is relative to the notebooks/ directory; adjust if running from repo root
data_path = Path('../data/raw/finwise_loan_applications.csv')
data = load_loan_data(data_path)

print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

In [ ]:
# Data types and non-null counts
data.info()

In [ ]:
# Statistical summary — numeric and categorical
data.describe(include='all').T

---
## 2. Data Cleaning

The cleaning step:
- Strips whitespace from column names
- Drops exact duplicate rows
- Raises an error if any numeric column contains missing values (fail-fast policy)
- Validates that `default_flag` is strictly binary {0, 1}

In [ ]:
data = clean_loan_data(data)

print(f"Rows after cleaning : {len(data):,}")
print(f"Missing values total: {data.isna().sum().sum()}")
print("\nTarget distribution:")
print(data['default_flag'].value_counts(normalize=True).rename('share').apply(lambda x: f"{x:.1%}"))

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Class balance and credit score by outcome ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(
    data=data, x='default_flag', ax=axes[0],
    palette=['#2f6f95', '#d46a4c'], hue='default_flag', legend=False
)
axes[0].set_title('Default class balance')
axes[0].set_xlabel('Default flag (0 = no default, 1 = default)')
axes[0].set_ylabel('Applications')

sns.boxplot(
    data=data, x='default_flag', y='credit_score', ax=axes[1],
    color='#76a5af'
)
axes[1].set_title('Credit score by outcome')
axes[1].set_xlabel('Default flag')

plt.tight_layout()
plt.show()

In [ ]:
# --- Numeric feature distributions split by default outcome ---
numeric_cols = data.select_dtypes(include='number').columns.drop('default_flag').tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for flag, colour in zip([0, 1], ['#2f6f95', '#d46a4c']):
        axes[i].hist(
            data.loc[data['default_flag'] == flag, col],
            bins=30, alpha=0.6, color=colour,
            label=f'default={flag}'
        )
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

# Hide unused axes
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric feature distributions by default outcome', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Categorical feature counts by outcome ---
cat_cols = data.select_dtypes(exclude='number').columns.drop('application_id').tolist()

fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))

for ax, col in zip(axes, cat_cols):
    cross = pd.crosstab(data[col], data['default_flag'], normalize='index')
    cross.plot(kind='bar', ax=ax, color=['#2f6f95', '#d46a4c'], legend=False)
    ax.set_title(f'Default rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Default rate by categorical feature', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heat-map of numeric features ---
corr = data[numeric_cols + ['default_flag']].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation matrix — numeric features + target')
plt.tight_layout()
plt.show()

---
## 4. Feature Preparation

- `application_id` is dropped — it is an identifier, not a generalizable applicant attribute.
- Stratified 80/20 train/test split with `random_state=42` ensures the default rate is preserved in both splits.
- All transformers (StandardScaler, OneHotEncoder) are fitted **inside the pipeline on training data only**, preventing data leakage.

In [ ]:
X, y = split_features_target(data)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # preserve class ratio
)

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Numeric features : {numeric_features}")
print(f"Categorical features: {categorical_features}")

---
## 5. Model Preparation

A scikit-learn `Pipeline` with:
1. `ColumnTransformer` — StandardScaler on numeric, OneHotEncoder (ignore unknown) on categorical
2. `LogisticRegression` with `class_weight='balanced'` to handle class imbalance

In [ ]:
model = build_model_pipeline(numeric_features, categorical_features)
model.fit(X_train, y_train)
print("Model trained successfully.")
print(model)

---
## 6. Results and Evaluation

In [ ]:
metrics = evaluate_model(model, X_test, y_test)

print("=" * 40)
print("CHECKPOINT 1 — HELD-OUT TEST METRICS")
print("=" * 40)
for metric in ('roc_auc', 'gini', 'accuracy', 'precision', 'recall', 'f1'):
    print(f"  {metric:<12}: {metrics[metric]:.4f}")
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(np.array(metrics['confusion_matrix']))

In [ ]:
# --- Confusion matrix heat-map ---
import matplotlib.ticker as mticker
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=np.array(metrics['confusion_matrix']),
    display_labels=['No default', 'Default']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix — test set')
plt.tight_layout()
plt.show()

In [ ]:
# --- ROC curve ---
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name='Logistic Regression baseline')
ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_title(f"ROC Curve (AUC = {metrics['roc_auc']:.4f})")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 7. Conclusion

### What was accomplished
- Loaded and validated 1 000 loan applications against a required schema.
- Cleaned duplicates; confirmed zero missing values and a binary target.
- Explored class balance, numeric distributions, categorical default rates, and feature correlations.
- Built a leakage-safe scikit-learn pipeline: StandardScaler + OneHotEncoder + Logistic Regression.
- Evaluated on a stratified 20 % hold-out set.

### Key observations
- **ROC AUC > 0.90** and **Gini > 0.45** — the baseline comfortably meets the Checkpoint 1 performance target.
- `credit_score` and `dti_ratio` show the strongest separation between defaulters and non-defaulters.
- The `class_weight='balanced'` setting compensates for the class imbalance visible in the EDA.

### Next steps (future checkpoints)
1. Replace Logistic Regression with **XGBoost** and tune hyperparameters.
2. Add **SHAP** explainability for per-application risk factor reports.
3. Serve predictions via a **FastAPI** REST endpoint.
4. Build the **React** frontend dashboard.
5. Conduct a **bias audit** across gender and age sub-groups.

> **Note**: Held-out metrics here are development evidence only. Before making production claims, evaluate against an independently approved validation set.